### Topic 1: Why Ingestion Exists At All

### Concept, Intuition, Why It Exists

- An LLM only knows two things: what was baked into it during training, and whatever you put in the prompt right now. It has never seen your internal FD policy PDF, today's interest rates, or last week's SOP.
- **RAG** exists to close this gap: instead of retraining a model every time a document changes, you store documents searchably and retrieve relevant pieces at query time, then hand them to the LLM as context.
- **Ingestion is stage zero of RAG** — it takes raw, messy, heterogeneous files (text, CSV, Excel, JSON, PDF) and turns them into a uniform shape that chunking, embedding, and retrieval can all operate on identically.
- Mental model — "the new-product-name problem": the knowledge exists *somewhere* (a PDF on a shared drive), but the model can't use it because it was never **ingested** into anything searchable. Being true and being retrievable are two different things.
- **Garbage in, garbage out applies harder here than anywhere else in the pipeline.** Drop a table, mangle a date, duplicate a stale policy — every later stage (chunking, embedding, retrieval) inherits that damage silently.

### Where Ingestion Sits

Raw files → **Ingestion** (normalize into Documents) → Chunking → Embedding → Vector Store → Retrieval → LLM answer. Every stage downstream of ingestion assumes its input already looks like a `Document`. That assumption only holds because ingestion enforces it.

### How It's Implemented In This Project

- Five source types ingested: structured text reference files, CSV/Excel tables, JSON page exports from a conversion team, and PDFs.
- Each source type gets its own loader module, all returning the same `Document` shape — covered in detail in the next topic.

### Real-World Issues

- **Silent failure**: a broken loader returns an empty string instead of raising — looks like "no relevant info" rather than "ingestion broke."
- **Schema drift**: an upstream file's column/field names change with no notice; code that assumes the old shape breaks silently or returns missing data.
- **Scaling**: eager loading holds every Document in memory at once — fine for 20 files, a crash at 50,000.
- **Cost**: re-embedding unchanged content daily because there's no change-detection.
- **Security/PII**: customer records get embedded into a vector store with no access control inherited from the source system.
- **Encoding bugs**: mixed-language content breaks on a non-UTF-8 default encoding.
- **Monitoring**: nobody notices ingestion volume dropped 90% until a customer complains — track documents-loaded-per-source and alert on anomalies.

### Design Decisions & Trade-offs

- Normalize everything into one `Document` shape immediately, at the loader boundary, rather than letting downstream code branch on source type. Small friction at ingestion time, in exchange for zero format-awareness anywhere else, forever.

### Alternatives

- Hand-rolled loaders (this project) — full control, good for learning, fine for a small number of well-understood sources.
- LangChain/LlamaIndex loaders — community-maintained edge-case handling, more sources supported out of the box.
- Managed ingestion (Unstructured.io, cloud Document Intelligence) — enterprise scale, compliance needs, OCR/table quality beyond a one-off script.

### Common Mistakes

- Treating ingestion as a "run once" script instead of a recurring, idempotent job.
- Not versioning the ingestion contract — an upstream schema change degrades retrieval silently.
- Conflating "no exception thrown" with "data is actually correct."

### Lead-Level Interview Questions

**Q: Why is ingestion a separate stage from chunking, not combined?**
A: Separation of concerns — ingestion's job is "get raw bytes into a normalized shape regardless of source." Chunking's job is "split that text into retrieval-sized pieces." Combined, every loader needs to know about chunk sizes, and every chunking change risks touching every loader.

**Q: Retrieval quality silently degraded over a month. Where do you start debugging?**
A: Work backwards stage by stage, checking *output* not just "did it run." Sample raw ingested Documents first — non-empty, sensible? Then chunks — coherent, not cut mid-sentence? Then embeddings — did the model/version change? Most real degradations trace back to ingestion or chunking, because those stages have the least observability.

**Q: A teammate wants to skip the Document abstraction and pass raw strings everywhere. Response?**
A: Looks simpler for one loader and one consumer. Stops looking simpler the moment you have 4 source types and 3 consumers needing to know *which* source a piece of text came from. Without metadata traveling with text, you lose citation, debugging, and per-source access control.

### Hidden Concepts Worth Knowing

- **Idempotency**: ingestion should be safe to run twice on the same input and produce the same result.
- **ETL vs. ELT**: this pipeline is closer to ELT — load raw-ish Documents first, transform (chunk/embed) afterward — keeping the option to re-chunk later without re-reading every source file.
- **Data contracts**: any schema consumed from an upstream team is an implicit contract. Make it explicit rather than discovering breakage in production.

### Revision Summary

> Ingestion exists because LLMs don't know your private data. It's RAG's first stage: normalize heterogeneous raw files into one uniform shape so every downstream stage can be written once, regardless of source format. Get this stage wrong and every later stage silently inherits the damage.

### Topic 2: The Document Object Pattern

### Concept, Intuition, Why It Exists

- A **Document** is the single shape every loader agrees to return: text plus metadata about where that text came from.
- This is the same idea every major RAG framework implements (LangChain's `Document`, LlamaIndex's `Node`) — `page_content` for the text the LLM will actually read, `metadata` for everything *about* that text that isn't part of the text itself (source file, page number, row index, version).
- The project's earlier, simpler `{"text":, "source":}` dicts from the initial prototype were already a stripped-down version of this exact idea — this formalizes it with richer metadata the earlier version was throwing away.
- **Why a plain dict and not a `@dataclass`/Pydantic model here?** Deliberate, for this stage: zero dependencies, trivially JSON-serializable, no class boilerplate. Trade-off: no compile-time type-checking, no validation on construction.

### Internal Working

- `make_document(page_content, metadata=None)` is the **only** place a Document gets constructed. Every loader calls this one function instead of building the dict inline, so the shape can never silently drift between loaders.
- `metadata` defaults to `{}`, never `None` — every downstream consumer does `doc["metadata"].get(...)`-style reads; defaulting to `None` would force a null-check everywhere instead of in exactly one place.

### Real-World Issues / Edge Cases

- **Schema evolution**: six months from now you add a `"language"` field to metadata. Documents ingested before that change simply won't have the key. Code must use `metadata.get("language", "unknown")`, never direct indexing — a defensive-read problem, not a reason to re-ingest everything.
- **Mutable default argument trap**: `metadata: dict = None` then `metadata or {}` is correct; writing `metadata: dict = {}` directly in the signature is a classic Python bug — the same dict object gets reused and mutated across calls.
- **Unbounded metadata growth**: nothing stops a loader from stuffing huge blobs into metadata. Vector stores often filter on metadata fields and may have payload size limits — keep metadata small and queryable, not a second copy of the content.

### Design Decisions & Trade-offs

- Plain dict, not a class: zero dependencies, fast to write — but no type safety, easy to typo a key with no error until runtime.
- One `make_document()` constructor used everywhere: shape can never silently drift — slight indirection cost, must remember to use it.
- `metadata` defaults to `{}` not `None`: downstream never null-checks — must remember `or {}`, not `= {}`, in the signature.

### Alternatives & When To Use Each

- `@dataclass` — once the project graduates past prototyping: IDE autocomplete, type-checking, typo'd field names fail immediately instead of silently returning `None`.
- Pydantic model — when you need runtime *validation*: e.g. rejecting empty `page_content`, enforcing `metadata["source"]` always present.
- LangChain's `Document` class — when already depending on LangChain for chunking/retrieval, to avoid an adapter layer.

### Common Mistakes & Production Failures

- Building Documents with ad hoc dicts in *some* loaders and via `make_document()` in others — the shape silently diverges across sources.
- Assuming every Document has every metadata key. A consumer hard-coding `doc["metadata"]["page"]` crashes on a CSV-sourced Document, which has `"row"`, not `"page"`.

### Lead-Level Interview Questions

**Q: We standardized every loader on `{"page_content":, "metadata":}`. What does *not* doing this actually cost you?**
A: Without a shared shape, every downstream consumer needs source-format-specific branching logic. One schema change anywhere means hunting down every consumer individually, instead of changing one loader.

**Q: This system later swapped its entire upstream PDF-handling — from an in-house parser to consuming another team's JSON. What made that swap touch exactly one file?**
A: Every loader, regardless of source, always returned the same `Document` shape. Chunking, dedup, and the manifest never had any PDF-specific assumption baked in — they only ever consumed `page_content` + `metadata`.

**Q: When would you graduate from a dict-based Document to a Pydantic model?**
A: The moment you need compile-time/IDE safety as the codebase grows, or runtime validation — e.g. rejecting empty `page_content` before it reaches embedding — which a raw dict won't enforce.

### Hidden Concepts Worth Knowing

- This pattern is a specific instance of the **Adapter pattern**: each loader converts a format-specific representation into one common interface the rest of the system depends on.
- It's also the practical expression of the **Dependency Inversion Principle** — downstream stages depend on an abstraction (`Document`), not on concrete source formats.

### Revision Summary

> Every loader returns `{"page_content": str, "metadata": dict}` via a single `make_document()` constructor. This is the one decision that makes the entire pipeline source-agnostic. Dict today, dataclass/Pydantic tomorrow once validation or type-safety actually matters.

In [1]:
"""
document.py
-----------
The single shared shape every loader in this pipeline returns.

A "Document" is just a dict with two keys:
  - page_content : str   -> the actual text
  - metadata     : dict  -> where it came from (source file, page no., row no., ...)

Why this exists: every downstream stage (dedup, chunking, embedding, search)
is written ONCE against this shape, regardless of whether the text originally
came from a .txt, .csv, .xlsx, or .json file. Change the source format,
and only the loader changes -- nothing downstream needs to know.
"""

from typing import Optional


def make_document(page_content: str, metadata: Optional[dict] = None) -> dict:
    """Build one Document. metadata defaults to an empty dict, never None,
    so downstream code can always safely do doc["metadata"].get(...)."""
    return {"page_content": page_content, "metadata": metadata or {}}


if __name__ == "__main__":
    doc = make_document("FD rate is 7.1% for 24 months.", {"source": "demo.txt"})
    print(doc)

{'page_content': 'FD rate is 7.1% for 24 months.', 'metadata': {'source': 'demo.txt'}}


## Topic 3: Text Loading

### Concept, Intuition, Why It Exists

- Targets: `fd_keyword_groups.txt`, `fd_negation_phrases.txt` — small, structured **reference** files, not prose documents.
- The defining decision: **load the whole file as ONE Document, never split it.** A keyword group only means something as a complete list — chunking it would tear related keywords apart and actively *hurt* retrieval, not help it. This is the "don't split this" exception to the usual rule.

### Internal Working

1. Open the file with **explicit UTF-8 encoding** — mixed-language terms will corrupt silently on a non-UTF-8 default.
2. Read the entire file content in one call.
3. Wrap it in exactly one `make_document()` call, with `metadata={"source": file_path}`.
4. A lazy generator yields that single Document; an eager version simply materializes the generator into a list.

### Real-World Issues, Edge Cases, Debugging

- **Encoding errors**: a file saved in `cp1252`/`latin-1` raises `UnicodeDecodeError` or — worse — silently produces garbled-but-valid-looking text if decoded with the wrong codec. Always pin `encoding="utf-8"` explicitly.
- **Empty file**: returns a Document with empty content, no error. Silently degrades retrieval — worth an explicit length check before accepting the Document.
- **Very large text files**: this loader assumes "small enough to read fully into memory" is safe — true for keyword lists, false if someone later points it at a huge log file.

### Design Decisions & Trade-offs

- Whole-file-as-one-Document vs. line-by-line: line-by-line would let you retrieve individual keyword groups, but these files are small enough that the entire file in one retrieval result is *more* useful than multiple stitched hits.
- This is a genuine granularity trade-off repeated throughout ingestion — the right granularity is "the smallest unit that's still semantically complete on its own," and that unit differs per source type.

### Alternatives & When To Use Each

- Whole file as one Document — small structured reference files where splitting destroys meaning.
- Line-by-line Documents — log files, line-delimited records where each line stands alone.
- Section-based splitting (by header markers) — larger structured text files with internal sections worth retrieving independently.

### Common Mistakes & Production Failures

- Reflexively running every source through the same chunker "for consistency," including small reference files that should never be split.
- Forgetting `encoding="utf-8"` — works fine locally, breaks in CI/production running a different default locale.

### Lead-Level Interview Questions

**Q: Why does this loader explicitly avoid chunking, when chunking is applied almost everywhere else?**
A: Chunking exists to keep retrieval units small and semantically coherent. For these files, the whole file already is one coherent unit — a keyword group is meaningless torn in half. Chunking here would optimize for chunk size at the expense of the actual goal, semantic completeness.

**Q: What's the actual failure mode of forgetting file encoding, and why might it pass review but fail in production?**
A: Locally, your editor/OS default likely matches the file's actual encoding, so it "just works." In production — a different container or locale — the default can differ, causing either a hard crash or silent corruption. Classic "works on my machine" bug, which is why explicit `encoding="utf-8"` is non-negotiable.

### Hidden Concepts Worth Knowing

- **BOM (Byte Order Mark)**: a few invisible bytes some editors prepend to UTF-8 files. `encoding="utf-8"` includes the BOM as a literal character; `encoding="utf-8-sig"` strips it.
- **Generators vs. lists**: trivial for a single file, but this establishes the convention every other loader follows — which matters a lot at scale.

### Revision Summary

> Text loading reads small, structured reference files as exactly ONE Document each — never split, because splitting would destroy the only thing that makes them useful. Always open with explicit `encoding="utf-8"`. This is the exception that proves chunking is a deliberate choice, not a default reflex.

In [2]:
"""
text_loader.py
---------------
Loads small structured reference .txt files (fd_keyword_groups.txt,
fd_negation_phrases.txt) as ONE Document per file.

Design choice: do NOT split these into multiple Documents. A keyword group
or a negation list only makes sense as a complete block -- chunking it would
tear related keywords apart and actively hurt retrieval quality.
"""

from document import make_document


def lazy_load_text(file_path: str):
    """Generator version -- reads the whole file once, yields exactly one Document."""
    with open(file_path, encoding="utf-8") as f:
        yield make_document(page_content=f.read(), metadata={"source": file_path})


def load_text(file_path: str) -> list:
    """Eager version -- same as lazy_load_text but returns a list, not a generator."""
    return list(lazy_load_text(file_path))


if __name__ == "__main__":
    docs = load_text("../data/fd_keyword_groups.txt")
    print(f"Loaded {len(docs)} document (whole file as one chunk)")
    print(f"  metadata     : {docs[0]['metadata']}")
    print(f"  content[:80] : {docs[0]['page_content'][:80]!r}")

Loaded 1 document (whole file as one chunk)
  metadata     : {'source': '../data/fd_keyword_groups.txt'}
  content[:80] : '# FD_provider — FD-signal keyword groups ONLY.\n# 4 groups, all positive FD-domai'


## Topic 4: CSV Loading

### Concept, Intuition, Why It Exists

- Targets: `fd_master_database.csv`, `fd_dataset_messy.csv` — tabular data where **each row is a complete, independently meaningful record** (one customer's FD, or one customer email).
- The defining decision, opposite of text loading: **one Document PER ROW**, not one Document for the whole file. A search for "Shobha Chopra's FD status" should retrieve her row, not the entire table dumped as one undifferentiated blob.

### Internal Working

1. Open the CSV with a dict-style reader — each row becomes a dict of `{column_name: value}`, using the header row as keys automatically.
2. For each row, join every column into a single text block as `"column: value"` lines — this is what becomes `page_content`.
3. Wrap it via `make_document()`, recording `{"source": file_path, "row": i}` in metadata — the row index lets you trace any Document straight back to its exact position in the source file for debugging.
4. A lazy generator yields one Document per row without holding the whole file in memory; the eager version materializes the full list.

### Real-World Issues, Edge Cases, Debugging, Security

- **`newline=""` is mandatory** when opening CSVs on some platforms — omitting it can cause extra phantom blank rows.
- **Type coercion landmines**: a dict-style reader returns every value as a **string**. Reading the same CSV through a library with type inference *without* forcing string dtype would silently turn a leading-zero account number into an integer, losing the leading zero permanently. The same discipline that protects this in the database layer must apply at ingestion too.
- **PII in page_content**: every row here includes customer name, mobile number, account number — real(-shaped) PII embedded directly into a vector store. This needs the same access control as the source database; "it's just a vector store" is not a security exemption.
- **Row order drift**: if the source CSV gets re-sorted or re-generated between ingestion runs, the row index no longer points to a stable identity — prefer a true primary key in metadata wherever the source has one.
- **Malformed rows**: a row with a stray unescaped comma can silently misalign columns depending on quoting — validate row length against header length before trusting a row.

### Design Decisions & Trade-offs

- One Document per row vs. one Document per N rows (batched): per-row gives maximum retrieval precision at the cost of many small Documents (more embedding calls). Batching trades precision for fewer, cheaper calls — reasonable for very large tables where row-level granularity isn't actually needed.
- Joining columns as `key: value` text vs. a templated sentence: `key: value` is simple, lossless, works for any schema without a template per table. A templated sentence often embeds *better* semantically but requires hand-maintenance and breaks silently when columns are added.

### Alternatives & When To Use Each

- One Document per row — row = independently meaningful record; need precise per-record retrieval.
- Templated natural-language sentence per row — want better embedding-similarity match to natural questions; willing to maintain a template.
- One Document per whole table — table is small and is always queried/returned as a whole, never per-record.
- Plain structured filtering, no embedding at all — query pattern is always exact-match/range-filter, never semantic; a real DB query beats vector search here.

### Common Mistakes & Production Failures

- Reading without forcing string dtype and silently corrupting leading-zero IDs or phone numbers — a bug that won't surface until someone notices a specific customer lookup failing.
- Embedding obvious PII into a vector store with the same access permissions as a public document set.
- Missing the `newline=""` requirement and getting phantom empty rows that inflate document counts and pollute dedup/embedding costs.

### Lead-Level Interview Questions

**Q: Why one Document per row instead of one per file, given text loading made the opposite choice?**
A: The granularity decision always follows "what is the natural, complete retrieval unit for this content?" For keyword files, that unit is the whole file. For a customer record table, that unit is one row — a query about one customer should retrieve exactly that customer's record, not the entire table or a fragment crossing two unrelated customers.

**Q: This CSV contains real PII columns. Walk through what changes between "fine for a notebook demo" and "fine for production."**
A: PII fields shouldn't be embedded in plaintext without justification — consider redaction for fields not needed for semantic search; the vector store needs the same RBAC as the source database; you need audit logging on what was retrieved and by whom; and deleting a customer's row from the source must also delete their corresponding vector — which a naive ingestion pipeline does not handle by default.

**Q: A type-coercion bug silently turned an account number into an integer during ingestion. How would you have caught this before production?**
A: A schema/contract check at ingestion time — assert each expected string column actually came back as a string matching an expected format, rather than trusting the reader's type inference. Validate structurally at the boundary, don't assume the reader did the right thing.

### Hidden Concepts Worth Knowing

- **Dict-style CSV reading vs. a dataframe library**: the dict-style reader is dependency-free and streams row-by-row naturally, good for the generator pattern. A dataframe library is faster for large files and offers richer dtype control, but reads more eagerly by default.
- **Quoting and escaping**: CSV's biggest hidden complexity is fields containing the delimiter itself — correctly quoted CSVs handle this fine by default, but hand-edited or badly-exported CSVs frequently don't, producing silently misaligned columns.

### Revision Summary

> CSV loading turns every row into its own Document via `key: value` text joining, because a row is the natural complete retrieval unit for tabular records. Pin string types for ID-like columns to avoid silent leading-zero corruption. PII in `page_content` means the vector store inherits the source DB's security requirements.

In [3]:
"""
csv_loader.py
--------------
Loads tabular files (fd_dataset_messy.csv, fd_master_database.csv) as ONE
Document per ROW. Every column is folded into page_content as "key: value"
lines, and the row index is preserved in metadata so any later debugging
can trace a Document straight back to its row in the source file.

Design choice: row-level granularity, not whole-file. A customer's FD
record is the natural retrieval unit -- you want a search to return "this
one customer's record", not "the entire 20,000-row table".
"""

import csv
from document import make_document


def lazy_load_csv(file_path: str):
    """Generator version -- reads row by row, never holds the whole file in memory."""
    with open(file_path, encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)          # each row -> {column_name: value}
        for i, row in enumerate(reader):
            content = "\n".join(f"{k}: {v}" for k, v in row.items())
            yield make_document(
                page_content=content,
                metadata={"source": file_path, "row": i},
            )


def load_csv(file_path: str) -> list:
    """Eager version -- use only when the file is small enough to fit in memory."""
    return list(lazy_load_csv(file_path))


if __name__ == "__main__":
    docs = load_csv("../data/fd_master_database.csv")
    print(f"Loaded {len(docs)} rows as Documents")
    print(f"  Document 0 metadata     : {docs[0]['metadata']}")
    print(f"  Document 0 content[:120]: {docs[0]['page_content'][:120]!r}")

Loaded 20 rows as Documents
  Document 0 metadata     : {'source': '../data/fd_master_database.csv', 'row': 0}
  Document 0 content[:120]: 'FD_No: BJ2019FD7717\nCustomer_Name: Shobha Chopra\nMobile_Number: 9686667744\nAccount_Number: 81960013389083\nFD_Amount_INR:'


## Topic 5: JSON Loading

### Concept, Intuition, Why It Exists

- Target: `related_documents_json/*.json` — output of an external conversion/OCR team's pipeline, one JSON file per source PDF, one entry per page inside it.
- **Why this loader exists instead of an in-house PDF parser**: parsing PDFs well (tables, scanned pages, OCR, encrypted files) is a genuinely hard, specialized problem — hard enough that a separate team owns it centrally. Consuming their JSON means there's exactly one source of truth for "what does this PDF actually say," instead of two independent PDF-parsing code paths that could silently disagree on the same source file.
- This loader's job is narrow and deliberate: read what the conversion team produced and map it into Documents. It does **not** judge OCR quality, confidence thresholds, or scanned-page detection — that responsibility stays with the team that owns the conversion pipeline.

### Internal Working

1. Read and parse the JSON file.
2. For each entry in the document's page list, build one Document: the page's text becomes `page_content`; `metadata` carries source filename, document code, version, and page number — exactly the provenance needed to trace a retrieved chunk back to a specific page of a specific document version.
3. A folder-level loader walks every `.json` file in a directory and concatenates their Documents — the entry point a scheduled ingestion job would actually call.

### How It's Implemented In This Project

- Loading one JSON file returns one Document per page.
- Loading the whole folder returns every page across every source document in one flat list, each still traceable to its exact source/page via metadata.
- **What swapping PDFs for JSON proved**: switching the entire upstream source from PDFs-in to JSON-in required changing exactly one file. Nothing downstream — dedup, the incremental manifest, chunking — needed to know anything changed, because the Document shape never changed. The manifest now hashes `.json` files instead of `.pdf` files with zero logic changes, proof it was never PDF-specific in the first place.

### Real-World Issues, Edge Cases, Debugging, Monitoring

- **Trusting upstream content without question is a real, named risk** — if the conversion team's pipeline produces a page with wrong text (e.g. OCR'd the wrong page, or pages swapped during conversion), this loader has no way to catch it, by design. That's the trade-off of not owning quality control: you gain simplicity and a single source of truth, you lose independent verification.
- **Missing/renamed fields**: if the conversion team renames a field or stops emitting one, reading it with a safe default returns `None` silently instead of raising — metadata quietly goes missing across every newly ingested document with no error anywhere. Worth a lightweight assertion on the fields you actually depend on, even without taking on quality judgment.
- **Empty pages**: a page with empty or near-empty text loads as a valid, near-useless Document with no signal that anything is wrong. Since this isn't your responsibility to validate quality-wise, the practical stance is: ingest what's given, and rely on the conversion team's own monitoring to catch extraction failures upstream — not duplicate that effort here.
- **New dependency risk**: this pipeline now depends on another team's pipeline being timely and correct. If their conversion job is delayed, you have stale or missing data with no PDF fallback.

### Design Decisions & Trade-offs

- Gained by consuming JSON instead of parsing PDFs in-house: a simpler pipeline with no OCR/PDF-parsing code to maintain, the benefit of a team that's better at PDF parsing than a one-off loader, and one clear source of truth for "what does this PDF say."
- Given up: independent verification against the source PDF, and a dependency on someone else's pipeline being timely and correct — extraction quality control now sits entirely with the upstream team.
- This is a **separation-of-concerns decision**, not a shortcut. Keeping an in-house parser *and* consuming the conversion team's JSON would mean two independent code paths that could extract different text from the same PDF — a worse outcome than trusting one well-maintained source.

### Alternatives & When To Use Each

- Consume a central conversion team's JSON (this project) — a specialized team already owns PDF/OCR parsing; you want one source of truth and want to stay out of that responsibility.
- In-house PDF parsing — no central conversion pipeline exists; you need full control over extraction timing and are willing to own quality.
- Hybrid fallback — in-house parsing only if the JSON is missing/stale, accepting the two-source-of-truth risk this design avoids, in exchange for resilience.

### Common Mistakes & Production Failures

- Silently swallowing a missing field with a default and never noticing an entire metadata dimension has gone missing across a batch of documents.
- Re-implementing quality checks that duplicate effort the upstream team already owns, blurring a deliberately drawn responsibility boundary.
- Assuming "JSON parsed without error" means "the content is correct" — parsing success and content correctness are different claims.

### Lead-Level Interview Questions

**Q: Why trust an external team's JSON instead of parsing PDFs yourself, when you clearly could?**
A: PDF/OCR parsing is a deep, specialized problem better solved once, centrally, by a team focused on it. Keeping a parallel in-house parser creates two sources of truth for the same PDF that could silently diverge. The trade is accepting a dependency on their pipeline's timeliness and correctness, in exchange for simplicity and not duplicating ownership of a problem that isn't yours to solve.

**Q: This loader doesn't validate OCR confidence or detect scanned pages. Isn't that a gap?**
A: It's a deliberately drawn boundary, not an oversight. Quality control for extraction belongs to the team that owns extraction — duplicating that logic here would mean maintaining two quality bars that can drift apart. The real safeguard is making sure that team's monitoring is solid, not re-implementing their checks downstream.

**Q: A field your metadata depends on gets silently renamed upstream. How would you catch that without taking on full validation responsibility?**
A: A lightweight assertion or schema check on just the fields you actually consume — not a quality judgment on content, just a contract check that the shape you depend on hasn't changed. That's a narrower, fair scope: you're allowed to defend your own contract without taking ownership of someone else's quality bar.

### Hidden Concepts Worth Knowing

- **Data contracts as a first-class artifact**: the JSON shape consumed here is an implicit contract between two teams. Lead-level practice makes contracts explicit (a shared schema, a versioned spec) so breaking changes are caught early, not discovered in production.
- **Drawing responsibility boundaries deliberately**: knowing what *not* to validate is as much a design decision as knowing what to validate — over-owning someone else's quality bar creates duplicated, drifting logic.

### Revision Summary

> JSON loading consumes a conversion team's per-page JSON output as the single source of truth for PDF content, instead of maintaining a competing in-house PDF parser. It maps pages to Documents and carries provenance metadata, but deliberately does not judge extraction quality — that responsibility stays with the team that owns conversion. The Document abstraction is what let this entire upstream source swap (PDF → JSON) touch exactly one file.

In [4]:
"""
json_loader.py
----------------
Loads the conversion/OCR team's JSON output (one JSON file per source
document, one entry per page) as ONE Document per PAGE.

Expected JSON shape (matches the conversion team's contract):
{
  "source_pdf": "02_FD_Product_Guide.pdf",
  "document_code": "FD-PROD-02",
  "version": "1.0",
  "pages": [
    {"page_number": 1, "text": "..."},
    {"page_number": 2, "text": "..."}
  ]
}

Why this loader exists at all: it replaces an in-house PDFLoader. Parsing
PDFs (tables, scanned pages, OCR) is a hard, specialized problem already
solved -- once, centrally -- by the conversion team. Consuming their JSON
means this pipeline has exactly ONE source of truth for "what does this
PDF actually say", instead of two PDF-parsing code paths that could
silently disagree.

NOTE: this loader deliberately does NOT judge OCR/extraction quality
(confidence thresholds, scanned-page detection, etc.) -- that is owned by
the conversion/OCR team, not this pipeline.
"""

import os
import glob
import json
from document import make_document

JSON_FOLDER = "../data/related_documents_json/"


def lazy_load_json(file_path: str):
    """Generator version -- yields one Document per page entry in the JSON."""
    with open(file_path, encoding="utf-8") as f:
        data = json.load(f)

    for page in data.get("pages", []):
        yield make_document(
            page_content=page.get("text", ""),
            metadata={
                "source": data.get("source_pdf", file_path),
                "document_code": data.get("document_code"),
                "version": data.get("version"),
                "page": page.get("page_number"),
            },
        )


def load_json(file_path: str) -> list:
    return list(lazy_load_json(file_path))


def load_all_json(folder: str = JSON_FOLDER) -> list:
    """Loads every .json file in a folder, concatenating all their Documents."""
    documents = []
    for path in sorted(glob.glob(os.path.join(folder, "*.json"))):
        documents.extend(load_json(path))
    return documents


if __name__ == "__main__":
    all_docs = load_all_json()
    print(f"Loaded {len(all_docs)} total pages across all JSON files")
    sources = sorted(set(d["metadata"]["source"] for d in all_docs))
    print(f"  Sources: {sources}")

Loaded 17 total pages across all JSON files
  Sources: ['01_FD_FAQ.pdf', '02_FD_Product_Guide.pdf', '03_FD_SOPs.pdf', '04_FD_Policy_Reference.pdf']


## Topic 6: Duplicate Documents — Hash + Near-Duplicate Detection

### Concept, Intuition, Why It Exists

- A real production scenario: `FD_Policy.pdf`, `FD_Policy_Final.pdf`, `FD_Policy_V2.pdf` all sitting in the same source folder — three filenames, but the actual content might be byte-identical in two cases and a reworded near-copy in the third. Ingesting all three blindly means the same fact gets embedded multiple times: wasted storage, wasted embedding cost, and retrieval returning near-identical chunks for one query, crowding out genuinely different relevant content.
- Two fundamentally different kinds of duplication need two fundamentally different detection tools: **exact duplicates** (byte-for-byte identical text, different filename — trivial and cheap to detect), and **near duplicates** (the same fact, reworded — requires understanding meaning, not just bytes, which is exactly what embeddings are for).

### Internal Working

**Stage 1 — Hash (cheap, exact, runs on everything):**
1. Compute a SHA-256 hash of each Document's text.
2. Track which hash was seen first while iterating every Document once.
3. If a hash repeats, that Document is an exact duplicate of the first one seen with that hash — recorded immediately, no embedding needed.
4. Anything that does not match an existing hash is kept for Stage 2.

**Stage 2 — Cosine similarity (expensive, near, runs only on survivors):**
1. Only the Documents that survived Stage 1 get embedded — this ordering is the entire point: embeddings cost real compute/money, so pay for them only on content you haven't already rejected for free.
2. Embed all surviving texts in one batch call, with vectors normalized to unit length.
3. Because vectors are normalized, cosine similarity reduces to a plain dot product — which is why normalization is requested explicitly at encode time rather than computing full cosine similarity separately.
4. Compare every surviving pair; any pair scoring at or above a similarity threshold (e.g. 0.97) is flagged as a near duplicate, with a "already flagged" tracker preventing a document from being reported as a duplicate of more than one earlier document.

### How It's Implemented In This Project

- Tested against the exact scenario this topic opened with: a small test set simulating `FD_Policy.json` / `FD_Policy_Final.json` / `FD_Policy_V2.json` correctly caught the byte-identical pair via hash and the reworded pair via cosine similarity — two different kinds of duplication, two different tools, applied in cost order.

### Real-World Issues, Edge Cases, Debugging, Scaling, Cost

- **Pairwise comparison in Stage 2 scales quadratically** with the number of post-hash-dedup Documents. Fine for hundreds of documents; a real problem at tens of thousands. Production systems use approximate nearest neighbor indexes to find near-duplicate candidates much faster instead of comparing every pair.
- **Threshold sensitivity**: the similarity threshold is a judgment call with real consequences in both directions — too low, and genuinely different content with similar phrasing gets wrongly merged or dropped; too high, and obvious near-duplicates slip through. There is no universally correct threshold; it should be tuned against a labeled sample of true positives/negatives for your actual content, not picked once and forgotten.
- **Which copy survives matters**: flagging duplicates doesn't decide which one to keep. A real ingestion pipeline needs an explicit policy — keep the one from the most recently modified file, keep the one with the higher document version, prefer a trusted source folder — because keeping an outdated policy version over a current one is a correctness bug, not just a storage-efficiency issue.
- **Cost accounting**: hashing is essentially free. Embedding is the real cost driver — this is precisely why the hash stage exists as a gate in front of the embedding stage, not as a parallel independent check.
- **Cross-batch duplicates**: this approach only catches duplicates within the list passed to it in one call. A document ingested today that duplicates one ingested last month won't be caught unless the comparison set includes historical embeddings too — meaning real production dedup needs to compare against the existing vector store, not just the current ingestion batch.

### Design Decisions & Trade-offs

- Cost-ordering the two stages is the central design decision: cheapest and most certain check first, most expensive and fuzziest check last, and only on what survives. Reversing the order would still work correctly but would pay embedding cost on documents you'd have rejected for free, strictly worse with no compensating benefit.
- Greedy first-match assignment vs. clustering: this approach greedily assigns each near-duplicate to the first earlier document it matches above threshold, not necessarily its closest match, and doesn't form true duplicate clusters when more than two documents are mutually similar. A clustering approach is more correct for that case but adds real implementation complexity for a problem that, at small scale, doesn't yet need it.

### Alternatives & When To Use Each

- Hash plus brute-force cosine comparison — small-to-medium document counts, simplicity over scale.
- Hash plus an approximate nearest neighbor index for near-duplicate search — tens of thousands or more documents, where pairwise comparison becomes a real bottleneck.
- Locality-sensitive hashing (MinHash/SimHash) — very large-scale near-duplicate detection where even approximate embedding comparison is too expensive, trading semantic understanding for raw speed.
- Vector-store-native dedup at insert time — production systems that already query the vector store on every ingest, checking for near-neighbors as part of the insert rather than as a separate batch pass.

### Common Mistakes & Production Failures

- Running the expensive embedding-based check on every document regardless of the hash check — silently multiplying embedding cost for zero added benefit on documents the hash check would already have caught for free.
- Picking a near-duplicate threshold once during development on a small test set and never revisiting it as real, messier production data starts flowing through.
- Deduplicating without a policy for which duplicate to keep, then discovering an outdated policy document was the one that survived.
- Forgetting that dedup needs to run against the existing vector store too, not just within a single day's ingestion batch — otherwise duplicates accumulate slowly across many ingestion runs instead of being caught once.

### Lead-Level Interview Questions

**Q: Why run the hash check before the embedding check, rather than doing both independently and combining results?**
A: Cost ordering. Hashing is essentially free; embedding costs real compute/money per document. Running hash first means embeddings are only computed for documents that survived the free check — hash-duplicates are a strict subset of what cosine similarity would also catch, since identical text has similarity 1.0, so there's zero correctness benefit to paying for embeddings on them.

**Q: This dedup approach only compares documents within a single ingestion batch. What's the actual risk in production, and how would you fix it?**
A: Duplicates that span ingestion runs go uncaught, because the comparison set never includes historical data. The fix is comparing new documents against the existing vector store at insert time — a nearest-neighbor query against what's already indexed, not just against the other documents in today's batch.

**Q: How would you choose and validate a near-duplicate similarity threshold for a real system, rather than picking a number because it "felt right"?**
A: Build a small labeled set of true near-duplicate pairs and true non-duplicate-but-similar pairs from real data, compute their similarity scores, and pick a threshold that separates them with acceptable false-positive/false-negative rates for your use case — then monitor in production and revisit as data distributions shift.

**Q: Two documents are flagged as near-duplicates, but only one is current policy and the other is stale. What does the dedup function do about that, and is that a gap?**
A: Nothing — it only flags the pair, it doesn't decide which to keep. That's a real gap for production use: an explicit "which copy wins" policy is required on top of dedup detection; treating "duplicate detected" as automatically solved would silently let a stale, wrong document win in some ordering-dependent way.

### Hidden Concepts Worth Knowing

- **Why normalized embeddings turn cosine similarity into a dot product**: cosine similarity divides the dot product by the product of both vectors' magnitudes. If both vectors are already unit-length, that denominator becomes 1 and the formula collapses to just the dot product — normalizing at encode time is what makes this simplification valid.
- **SHA-256 collision risk is not a practical concern here**: the probability of two different texts producing the same hash is astronomically small — "same hash" can be treated as "same content" without hedging.
- **Locality-Sensitive Hashing (LSH)**, including MinHash and SimHash, is the broader family of techniques that make near-duplicate detection scale past brute-force pairwise comparison — worth knowing by name even if current scale doesn't yet require it.

### Revision Summary

> Two-stage dedup, cheapest-and-most-certain first: a content hash catches exact duplicates for free across every document; cosine similarity (via normalized-embedding dot products) catches reworded near-duplicates, but only on documents that survived the hash check, because embeddings cost real money. Pairwise comparison is the scaling limit — approximate nearest neighbor indexes are the production answer past a few thousand documents. Flagging duplicates is not the same as resolving which copy to keep — that needs an explicit policy.

In [5]:
"""
dedup.py
---------
Two-stage duplicate detection for Documents coming from different filenames
(e.g. FD_Policy.json / FD_Policy_Final.json / FD_Policy_V2.json).

Stage 1 - HASH (cheap, exact):
    SHA-256 of the text. Identical content -> identical hash, regardless
    of filename. Runs on EVERY document; cost is negligible.

Stage 2 - COSINE SIMILARITY (expensive, near/fuzzy):
    Only runs on documents that SURVIVED stage 1. Embeddings cost real
    money/compute, so we never pay for them on content we'd already reject.
    Catches reworded duplicates: same fact, different phrasing.

Cost-ordering is the whole design: cheapest, most certain check first;
expensive, fuzzy check only on what's left.
"""

import hashlib
import numpy as np

# Loaded lazily so importing this module doesn't force a model download
# just to use content_hash() alone.
_model = None


def _get_model():
    global _model
    if _model is None:
        from sentence_transformers import SentenceTransformer
        _model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    return _model


def content_hash(text: str) -> str:
    """A fingerprint for a piece of text. Identical text -> identical hash."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def find_duplicates(documents: list, near_duplicate_threshold: float = 0.97) -> list:
    """Returns a list of (doc_index, duplicate_of_index, 'exact' | 'near (score)')."""
    seen_hashes = {}          # hash -> index of the FIRST document with that hash
    kept_for_embedding = []   # (index, text) for docs that survived the hash check
    duplicates = []

    # ---- Stage 1: hash check (exact duplicates) ----
    for i, doc in enumerate(documents):
        h = content_hash(doc["page_content"])
        if h in seen_hashes:
            duplicates.append((i, seen_hashes[h], "exact"))
        else:
            seen_hashes[h] = i
            kept_for_embedding.append((i, doc["page_content"]))

    # ---- Stage 2: cosine similarity (near duplicates) ----
    if len(kept_for_embedding) > 1:
        indices, texts = zip(*kept_for_embedding)
        model = _get_model()
        embeddings = model.encode(list(texts), normalize_embeddings=True, show_progress_bar=False)

        already_flagged = set()
        for a in range(len(indices)):
            if indices[a] in already_flagged:
                continue
            for b in range(a + 1, len(indices)):
                if indices[b] in already_flagged:
                    continue
                score = float(np.dot(embeddings[a], embeddings[b]))  # vectors normalized -> dot == cosine
                if score >= near_duplicate_threshold:
                    duplicates.append((indices[b], indices[a], f"near ({score:.3f})"))
                    already_flagged.add(indices[b])

    return duplicates


if __name__ == "__main__":
    from document import make_document

    test_docs = [
        make_document("FD rate is 7.1 percent for 24 month tenure.", {"source": "FD_Policy.json"}),
        make_document("FD rate is 7.1 percent for 24 month tenure.", {"source": "FD_Policy_Final.json"}),
        make_document("FD rate is 7.1 percent for a 24 month tenure period.", {"source": "FD_Policy_V2.json"}),
        make_document("Premature withdrawal incurs a 1 percent penalty.", {"source": "FD_Policy.json"}),
    ]
    dupes = find_duplicates(test_docs, near_duplicate_threshold=0.85)
    for idx, of_idx, kind in dupes:
        print(f"  Document {idx} ({test_docs[idx]['metadata']['source']}) is a {kind} "
              f"duplicate of Document {of_idx} ({test_docs[of_idx]['metadata']['source']})")


  Document 1 (FD_Policy_Final.json) is a exact duplicate of Document 0 (FD_Policy.json)
  Document 2 (FD_Policy_V2.json) is a near (0.993) duplicate of Document 0 (FD_Policy.json)


## Topic 7: Incremental Ingestion

### Concept, Intuition, Why It Exists

- The naive approach — "every time we ingest, re-read, re-chunk, and re-embed every single source file" — works fine for a handful of demo files and breaks down hard at real scale: re-embedding thousands of unchanged documents daily wastes compute, money, and time on content that hasn't changed at all since yesterday.
- This is the same discipline `insert_fd_record()` / `update_fd_record()` already established in the FD database layer for the records table: a real database doesn't get dropped and rebuilt from scratch on every update — it tracks individual record changes and only touches what actually changed. Incremental ingestion applies that same principle to **files** instead of **rows**: a manifest remembers what's already been ingested, and only new or changed files get reprocessed.

### Internal Working

1. A manifest is a flat record mapping each tracked file path to the hash of its content and the timestamp it was last ingested.
2. Loading the manifest returns an empty record on the very first run — no manifest exists yet, so nothing has "been seen before."
3. Checking which files need ingestion computes each tracked file's current hash (of the raw file bytes, not just its text content — this hashes the whole file, catching any byte-level change) and compares it against what the manifest has recorded. A mismatch, or no entry at all, means "needs (re)ingesting"; a match means "skip, nothing changed."
4. Updating a manifest entry records the file's current hash and an ingestion timestamp — but only in memory at first.
5. Saving the manifest is the only step that actually writes to disk, and it happens once, after processing — this ordering has real consequences explored below.

### How It's Implemented In This Project

- Demonstrated as a deliberate before/after: a first run against an empty manifest returns every tracked JSON file as needing ingestion; after updating and saving the manifest, a second run against the same unchanged files returns an empty list. Tested directly by changing the content of exactly one tracked file between runs and confirming only that file reappears as needing ingestion — the unchanged files are never touched, which is the entire point, and the part most naive "just reload everything" pipelines get wrong.

### Real-World Issues, Edge Cases, Debugging, Scaling

- **Concurrent writers**: the manifest is a single file. Two ingestion jobs running concurrently and both saving results in last-write-wins — whichever job saves last silently overwrites the other's updates, with no error and no warning. At real scale, the fix is a proper datastore with atomic/transactional writes, not a flat file being read-modified-written by multiple processes.
- **Keyed on path, not content**: if a file gets renamed but its content is byte-identical, it's treated as brand new and fully re-ingested — a false positive, wasted compute on content that hasn't actually changed. Whether this is "wrong" is a genuine trade-off, not an obvious bug to fix: correcting it means also checking new files' hashes against every existing manifest entry's hash, not just the matching path, which adds real complexity for a case that may be rare depending on how often renames actually happen in your real source.
- **Crash recovery**: updating a manifest entry only mutates the in-memory record, file by file, inside a loop; saving runs once, after the loop finishes. A crash midway through processing several changed files means nothing gets persisted: on restart, all of them still look unchanged-since-last-save and correctly get reprocessed. This is safe — no data corruption, no partial/incorrect state — but wasteful, since every file gets redone even if most had already succeeded before the crash. The fix is saving the manifest incrementally, after each file, trading a bit of extra I/O for not having to redo already-completed work after a crash.
- **Embedding model version is not tracked**: the manifest tracks content hash only, not which embedding model produced the resulting vectors in the store. Swap embedding models, and the manifest happily reports "nothing to ingest" while every existing vector is now incomparable to new queries embedded with the new model — a real blind spot, not a minor one. A production manifest should also record the embedding model name/version per entry, and treat a model change as invalidating every existing entry.
- **Deletions are invisible**: if a source file is deleted from disk, the manifest still has a stale entry for it, and nothing in this design removes the corresponding vectors from the store — incremental ingestion as built here only ever adds or updates, never tombstones.

### Design Decisions & Trade-offs

- File-level granularity, not row/page-level: matches the granularity at which sources actually change in this project, but means a single-character change anywhere in a large file forces re-ingesting the entire file, not just the changed page or row. Acceptable when source files are small; would need finer-grained hashing for very large individual files where that waste becomes significant.
- A single flat manifest file vs. a real datastore: simplest possible implementation for a single-process, single-writer context — explicitly not safe for concurrent writers, which is named as a known limitation rather than silently ignored.
- Hashing whole file bytes vs. hashing extracted text content: hashing raw bytes catches any change, including ones that wouldn't affect extracted text. Hashing extracted text instead would be more precise about "did the actual content change" but requires re-parsing the file just to check whether parsing it again is even necessary — a chicken-and-egg cost that whole-file hashing avoids entirely.

### Alternatives & When To Use Each

- A flat manifest file, file-hash keyed — single-writer, small-to-medium file counts, simplicity prioritized.
- A real database table for the manifest — concurrent ingestion jobs, need transactional updates, want to query ingestion history.
- Content-addressable storage, where the hash of a piece of content is its own storage key — unifies deduplication and change-detection into one system, common in data lake architectures.
- Vector-store-native upsert using content hash as part of the vector identity — production RAG systems where the vector store itself should be the single source of truth for "is this already indexed," not a separate manifest file.
- Change Data Capture from a source database — when the source is a live database rather than flat files, streaming row-level changes in near-real-time instead of polling file hashes on a schedule.

### Common Mistakes & Production Failures

- Re-ingesting everything daily "to be safe," defeating the entire purpose of incremental ingestion and quietly burning embedding budget at scale.
- Never saving the manifest incrementally, so any mid-run crash forces a full redo of an entire batch instead of resuming from where it left off.
- Swapping an embedding model without realizing the manifest gives no signal that every existing vector is now stale relative to the new model.
- Treating an empty "files to ingest" result as proof the knowledge base is fully up to date, without separately verifying that file deletions are also being handled — they aren't, in this design.

### Lead-Level Interview Questions

**Q: The manifest is a single file. Two ingestion jobs run concurrently and both try to update it. What happens?**
A: Last write wins — whichever job saves last silently overwrites the other's updates, with no error and no warning. At real scale, the actual fix is a proper datastore with atomic/transactional writes, not a flat file being read-modified-written by multiple processes.

**Q: The manifest keys on file path, not content. A file gets renamed but the content is byte-identical. What happens, and is that the right behavior?**
A: It gets treated as brand new and fully re-ingested — a false positive, wasted compute on unchanged content. Whether that's "wrong" is a genuine trade-off: fixing it means hashing against the manifest's existing content hashes too, not just the matching path, which adds complexity for a case that may be rare in practice. The senior-level answer isn't "obviously fix it" — it's naming the trade-off and deciding based on how often renames actually happen in your real source.

**Q: If ingestion crashes halfway through processing several changed files, what state is the manifest left in, and does a restart resume correctly?**
A: Updating a manifest entry only mutates the in-memory record; saving — the only thing that writes to disk — runs once, after the loop finishes. A crash midway means nothing gets persisted: on restart, all files still look unchanged-since-last-save and correctly get reprocessed. Safe, but wasteful — every file gets redone even if most had already succeeded. The fix is saving incrementally after each file, trading some I/O overhead for not having to redo already-completed work after a crash.

**Q: The manifest tracks content hash only, not which embedding model produced the resulting vectors. If you swap embedding models, does this manifest tell you anything changed?**
A: No — and that's a real blind spot. The manifest would happily report "nothing to ingest" while every existing vector in the store is now incomparable to new queries embedded with the new model. A production manifest needs to track embedding model identity/version per entry and treat a model change as invalidating every existing entry, forcing a full re-embed.

**Q: How does this manifest design handle a source file being deleted?**
A: It doesn't — this is a real gap. The manifest entry for the deleted file simply stays stale forever, and nothing removes the corresponding vectors from the vector store. A complete incremental-ingestion design needs a reconciliation step that compares the manifest's tracked paths against what currently exists on disk and removes vectors for anything no longer present.

### Hidden Concepts Worth Knowing

- **Idempotency**: incremental ingestion is what makes the whole pipeline safely re-runnable — running it twice on unchanged input produces the same end state as running it once, which is a property every production scheduled job should have.
- **Content-addressable storage**: a broader pattern where the hash of a piece of content is its own identifier or storage key — the same hash-as-identity idea this manifest uses for change detection, taken further as the actual storage mechanism.
- **Watermarking / cursor-based incremental processing**: an alternative incremental pattern common in data engineering — track "the last timestamp or offset processed" rather than per-item hashes — works well for append-only logs/streams but doesn't handle in-place file edits the way hash-based tracking does, which is why this project uses hashing instead.

### Revision Summary

> Incremental ingestion mirrors the `insert_fd_record()`/`update_fd_record()` discipline from the FD database, applied to files instead of rows: a manifest tracks each tracked file's content hash; only new-or-changed files get reprocessed; unchanged files are skipped entirely, saving re-chunking, re-embedding, and API cost. Known gaps to name explicitly in any review: not concurrency-safe, keyed on path not content, no embedding-model-version tracking, and no deletion handling.

In [6]:
"""
incremental_ingestion.py
--------------------------
Skip re-embedding files that haven't changed since the last ingestion run --
the same discipline insert_fd_record()/update_fd_record() already established
for the FD database (fd_database.py), applied to source files instead of rows.

A "manifest" (a JSON file) remembers, for every tracked source file:
    { file_path: {"hash": <sha256 of file bytes>, "last_ingested": <UTC ISO timestamp>} }

Each run compares CURRENT file hashes against the manifest. Only NEW or
CHANGED files get (re)ingested -- unchanged files are skipped entirely:
no re-chunking, no re-embedding, no wasted API/embedding cost.

Known limitations (see theory section for the full discussion):
  - Single JSON file -> not safe for concurrent writers (last write wins).
  - Keyed on file PATH, not content -> a rename looks like "new file".
  - Manifest is updated in-memory and only persisted via save_manifest() --
    a crash mid-run loses ALL progress for that run (safe, but wasteful).
  - Does NOT track which embedding model produced existing vectors --
    swapping models silently leaves stale vectors with no warning.
"""

import os
import json
import hashlib
import glob
from datetime import datetime, timezone

MANIFEST_PATH = "ingestion_manifest.json"


def _file_hash(path: str) -> str:
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


def load_manifest(path: str = MANIFEST_PATH) -> dict:
    """Reads our 'memory' of what's already been ingested. Empty dict on first run."""
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_manifest(manifest: dict, path: str = MANIFEST_PATH) -> None:
    """Writes the manifest back to disk so the NEXT run can read it."""
    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)


def get_files_to_ingest(file_paths: list, manifest: dict) -> list:
    """Compares each file's CURRENT hash against the hash recorded last time.
    Different hash (or never seen before) = needs (re)ingesting."""
    to_ingest = []
    for path in file_paths:
        current_hash = _file_hash(path)
        if manifest.get(path, {}).get("hash") != current_hash:
            to_ingest.append(path)
    return to_ingest


def update_manifest_entry(manifest: dict, file_path: str) -> None:
    """Records 'this file, with this exact hash, was ingested just now'.
    Mutates manifest IN MEMORY ONLY -- call save_manifest() to persist."""
    manifest[file_path] = {
        "hash": _file_hash(file_path),
        "last_ingested": datetime.now(timezone.utc).isoformat(),
    }


if __name__ == "__main__":
    JSON_FOLDER = "../data/related_documents_json/"
    tracked_files = sorted(glob.glob(os.path.join(JSON_FOLDER, "*.json")))

    # ---- Run 1: everything is new ----
    manifest = load_manifest()
    to_ingest = get_files_to_ingest(tracked_files, manifest)
    print(f"Run 1 -- files to ingest: {[os.path.basename(f) for f in to_ingest]}")
    for f in to_ingest:
        update_manifest_entry(manifest, f)
    save_manifest(manifest)

    # ---- Run 2: nothing changed, expect empty list ----
    manifest = load_manifest()
    to_ingest = get_files_to_ingest(tracked_files, manifest)
    print(f"Run 2 -- files to ingest: {to_ingest}  (expect empty)")

Run 1 -- files to ingest: []
Run 2 -- files to ingest: []  (expect empty)
